> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB16 — GI donor-abundance audit: absolute counts versus population fractions

**Objective.** Test whether the high Day 10 target donor fraction in NB15 can be attributed solely to normalization, by comparing the **original culture-derived donor proxy** with the **NB15 estimated donor fraction**. The source consists of the previously audited 32 GI animal-day observations (16 animals measured at Days 10 and 14) and existing NB15 posterior summaries. This is **an audit, not a new Bayesian fit**.

**Key distinction.** The observational donor proxy is `N_SmTc − N_SmRfTc` (CFU/g). It can be negative when selective plate estimates disagree; no value is clipped or discarded. The Bayesian donor fraction is `D / (R + D + T2)`. The two measures have different units and meanings, and **the NB15 Day-shared model imposes a common group-level donor mean by day**; it does not force animal-specific D or the donor fraction to be equal.

**Purpose.** Compare the 6-hour PE results? **No.** This notebook concerns **GI Days 10 and 14 only**. It does not estimate the original inoculum per gram of faeces, a donor growth rate, an infection or conjugation rate, or a causal treatment effect. It does not resolve plate measurement noise, and a large donor proxy difference does not prove a biological difference in donor abundance.

**Instructions.** Run cells sequentially. Original animal-level inputs must be separately authorised and supplied; they are not included in this release.

**Outputs.** Summary CSVs and two PNG figures in `06_Results/NB16_GI_Donor_Audit`. Neither NB15 nor the manuscript is modified. **KWS-derived animal data: private; do not publish or redistribute without permission.**

In [ ]:
#@title NB16.01 — Set up analysis libraries
import io, hashlib, zlib, base64
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('Libraries ready; this notebook does not run Stan.')

In [ ]:
#@title NB16.02 — Connect Google Drive and select output folder
PROJECT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    mydrive = Path('/content/drive/MyDrive')
    direct = mydrive / 'Transmid_Animal_Bayesian_Project'
    if direct.is_dir():
        PROJECT = direct
    else:
        matches = [p for p in mydrive.rglob('Transmid_Animal_Bayesian_Project') if p.is_dir()]
        if len(matches) == 1:
            PROJECT = matches[0]
        elif len(matches)>1:
            raise RuntimeError(f'Ambiguous project folders: {matches}')
except ImportError:
    pass
if PROJECT is None:
    PROJECT = Path.cwd() / 'Transmid_Animal_Bayesian_Project'
    print('Project folder not found in Drive; using local folder:', PROJECT)
RESULT_DIR = PROJECT/'06_Results'/'NB16_GI_Donor_Audit'
DATA_DIR = PROJECT/'03_Analysis_Data'/'NB16_GI_Donor_Audit'
RESULT_DIR.mkdir(parents=True,exist_ok=True)
DATA_DIR.mkdir(parents=True,exist_ok=True)
print('Results:',RESULT_DIR)

In [ ]:
#@title NB16.03 — Load separately authorised NB15-derived results
# Public-release version: the original animal-level count audit is NOT embedded.
# Supply the following files in DATA_DIR only if permitted to access them:
# NB15_original_count_audit.csv, NB15_group_population_fractions.csv,
# NB15_donor_hierarchical_posterior.csv.
import pandas as pd
NEEDED={
 'audit':'NB15_original_count_audit.csv',
 'fractions':'NB15_group_population_fractions.csv',
 'donor':'NB15_donor_hierarchical_posterior.csv',
}
def authorised_csv(key):
    p=DATA_DIR/NEEDED[key]
    if not p.is_file():
        raise FileNotFoundError(f'Authorised NB15 input required: {p}. See README.')
    return pd.read_csv(p)
audit=authorised_csv('audit')
fractions=authorised_csv('fractions')
donor_posterior=authorised_csv('donor')
assert len(audit)==32
assert audit.groupby(['day','group']).size().eq(8).all()
assert audit['animal_id'].nunique()==16
assert set(audit['day'])=={10,14}
assert np.allclose(audit['raw_D_proxy_CFU_g'], audit['N_SmTc']-audit['N_SmRfTc'])
print('Separately supplied NB15 inputs validated (individual animal measurements not published).')


In [ ]:
#@title NB16.04 — Compare observed donor proxy and modeled fraction
GROUP_ORDER = [(10,'Target'),(10,'Non-target'),(14,'Target'),(14,'Non-target')]
rows=[]
fr=fractions[(fractions.scenario=='Baseline') & (fractions.component=='D')].copy()
for day,group in GROUP_ORDER:
    a=audit[(audit.day==day)&(audit.group==group)]
    label=f'Day {day} — {group}'
    f=fr.loc[fr.group==label].iloc[0]
    rows.append(dict(day=day,group=group,n_animals=len(a),
      donor_proxy_median_CFU_g=float(a.raw_D_proxy_CFU_g.median()),
      donor_proxy_min_CFU_g=float(a.raw_D_proxy_CFU_g.min()),
      donor_proxy_max_CFU_g=float(a.raw_D_proxy_CFU_g.max()),
      negative_proxy_count=int((a.raw_D_proxy_CFU_g<0).sum()),
      donor_fraction_posterior_mean=float(f.posterior_mean),
      donor_fraction_q025=float(f.q025),donor_fraction_q975=float(f.q975)))
summary=pd.DataFrame(rows)
summary.to_csv(RESULT_DIR/'NB16_donor_absolute_vs_fraction.csv',index=False)
show=summary.copy()
for c in ('donor_fraction_posterior_mean','donor_fraction_q025','donor_fraction_q975'):
    show[c]=100*show[c]
display(show.round(3))
ratios=[]
for day in [10,14]:
    s=summary[summary.day==day]
    ratio=(s.iloc[0].donor_proxy_median_CFU_g/s.iloc[1].donor_proxy_median_CFU_g)
    ratios.append({'day':day,'observed_median_donor_proxy_target_over_non_target':ratio})
ratio_df=pd.DataFrame(ratios)
ratio_df.to_csv(RESULT_DIR/'NB16_observed_donor_proxy_ratios.csv',index=False)
display(ratio_df.round(2))
print('Note: ratios summarize observed noisy count differences; they are NOT inferred donor growth ratios.')

In [ ]:
#@title NB16.05 — Plot observed animal donor-count proxies (including negatives)
labels=[f'D{d} '+('Target' if g=='Target' else 'Non-target') for d,g in GROUP_ORDER]
fig,ax=plt.subplots(figsize=(9,5.5))
for x,(day,group) in enumerate(GROUP_ORDER):
    sel=audit[(audit.day==day)&(audit.group==group)]['raw_D_proxy_CFU_g'].to_numpy(float)
    # Fixed offsets preserve reproducibility while showing each individual animal.
    dx=np.linspace(-0.16,0.16,len(sel))
    ax.scatter(np.repeat(x,len(sel))+dx,sel,s=32,alpha=.8)
    median=np.median(sel)
    ax.plot([x-.22,x+.22],[median,median],linewidth=2)
ax.axhline(0,linewidth=1)
ax.set_xticks(range(4),labels)
ax.set_yscale('symlog',linthresh=1e6)
ax.set_ylabel('Observed donor proxy (CFU/g; signed, symlog axis)')
ax.set_title('GI: original donor proxy by animal and day')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig.savefig(RESULT_DIR/'NB16_observed_D_proxy_by_animal.png',dpi=180,bbox_inches='tight')
plt.show()
print('Note: a negative estimate is an inconsistent measurement, not negative bacteria.')

In [ ]:
#@title NB16.06 — Plot NB15 normalized donor fractions with credible intervals
means=summary.donor_fraction_posterior_mean.to_numpy()*100
lo=summary.donor_fraction_q025.to_numpy()*100
hi=summary.donor_fraction_q975.to_numpy()*100
fig,ax=plt.subplots(figsize=(9,5.2))
ax.bar(labels,means,yerr=[means-lo,hi-means],capsize=4,alpha=.82)
ax.set_ylim(0,100)
ax.set_ylabel('NB15 donor fraction, D / (R + D + T2), %')
ax.set_title('GI: modeled normalized donor fractions (baseline NB15)')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig.savefig(RESULT_DIR/'NB16_NB15_normalized_D_fraction.png',dpi=180,bbox_inches='tight')
plt.show()
print('Bars are posterior MEANS; error bars are 95% credible intervals. These are not direct donor-count measurements.')

In [ ]:
#@title NB16.07 — Shared donor reference and interpretation checks
reference = donor_posterior[(donor_posterior.scenario=='Baseline') &
  (donor_posterior.quantity=='reference_D_CFU_g')][['day','median','q025','q975']]
print('NB15 day-shared log-donor reference exponentiated into CFU/g (not a group-specific animal median):')
display(reference)
neg_r=int((audit.raw_R_proxy_CFU_g<0).sum())
neg_d=int((audit.raw_D_proxy_CFU_g<0).sum())
print(f'Count audit: {neg_d}/32 negative donor proxies, {neg_r}/32 negative recipient proxies.')
assert neg_d==1 and neg_r==14
print('Day 10 donor proxy differs strongly between target statuses. A difference in D fraction is NOT solely a denominator effect in these observed proxies.')
print('NB15 nonetheless imposes a common group-level donor log mean per day; its posteriors cannot independently confirm equality.')
print('Biological explanations and plate measurement errors remain unresolved; do not infer treatment causality from these descriptive counts.')
print('No manuscript or earlier notebook was changed.')